# Google Drive Description Writeback Starter

This notebook reads confirmed names from a Google Sheet and writes those names back to matching Google Drive file descriptions.

Expected workflow:

1. The pipeline or a derived process creates a `Face Occurrences` sheet.
2. A user confirms names in a `Face Library` sheet.
3. This notebook joins confirmed names to Drive file links.
4. The notebook writes a managed metadata block into each Drive file description.

Keep `DRY_RUN = True` until the planned updates look correct.

## 1. Install Dependencies And Authenticate

In [ ]:
!pip install gspread google-api-python-client google-auth

from google.colab import auth
auth.authenticate_user()

import google.auth
creds, _ = google.auth.default()

print("Authenticated with Google.")

## 2. Configuration

Paste the Google Sheet ID below. The Sheet ID is the long value in a URL like:

`https://docs.google.com/spreadsheets/d/SPREADSHEET_ID/edit`

In [ ]:
SPREADSHEET_ID = "PASTE_GOOGLE_SHEET_ID_HERE"

FACE_LIBRARY_WORKSHEET = "Face Library"
FACE_OCCURRENCES_WORKSHEET = "Face Occurrences"

# Keep True until dry-run output looks correct.
DRY_RUN = True

# Column names expected in Face Library tab.
FACE_ID_COL = "Face ID"
CONFIRMED_NAME_COL = "Confirmed Name"
CONFIRM_STATUS_COL = "Confirm Status"
NOTES_COL = "Notes / Role"

# Column names expected in Face Occurrences tab.
DRIVE_LINK_COL = "Drive Link"
FILE_NAME_COL = "File Name"
MEDIA_TYPE_COL = "Media Type"
START_TIME_COL = "Start Time"
END_TIME_COL = "End Time"
CONFIDENCE_COL = "Confidence_this_person"

APPROVED_VALUES = {"yes", "y", "true", "approved", "confirmed"}

DESCRIPTION_START = "<unlabeled-media-tagger>"
DESCRIPTION_END = "</unlabeled-media-tagger>"

## 3. Imports And Helper Functions

In [ ]:
import json
import re
from datetime import datetime, timezone

import gspread
from googleapiclient.discovery import build


def parse_drive_file_id(value):
    """Parse Drive file ID from a Drive link or raw ID."""
    value = str(value).strip()
    if not value:
        return None

    patterns = [
        r"/file/d/([a-zA-Z0-9_-]+)",
        r"[?&]id=([a-zA-Z0-9_-]+)",
    ]

    for pattern in patterns:
        match = re.search(pattern, value)
        if match:
            return match.group(1)

    # Accept raw Drive IDs.
    if re.fullmatch(r"[a-zA-Z0-9_-]{20,}", value):
        return value

    return None


def strip_managed_block(description):
    """Remove old managed metadata block, preserving human-written text."""
    description = description or ""

    start = description.find(DESCRIPTION_START)
    end = description.find(DESCRIPTION_END)

    if start == -1 or end == -1 or end < start:
        return description

    end += len(DESCRIPTION_END)
    return f"{description[:start]}{description[end:]}".strip()


def build_description(existing_description, metadata):
    """Build new Drive description with managed JSON block."""
    base_text = strip_managed_block(existing_description)
    payload = json.dumps(metadata, sort_keys=True, separators=(",", ":"))
    block = f"{DESCRIPTION_START}{payload}{DESCRIPTION_END}"

    if base_text:
        return f"{base_text}\n\n{block}"

    return block


def parse_float_or_none(value):
    try:
        return float(value)
    except Exception:
        return None


def clean_row(row):
    """Normalize Google Sheet row values to stripped strings."""
    return {str(key).lstrip("\ufeff").strip(): str(value).strip() for key, value in row.items()}

## 4. Read Google Sheets

In [ ]:
def read_sheets(creds):
    gc = gspread.authorize(creds)
    spreadsheet = gc.open_by_key(SPREADSHEET_ID)

    face_library = spreadsheet.worksheet(FACE_LIBRARY_WORKSHEET).get_all_records()
    occurrences = spreadsheet.worksheet(FACE_OCCURRENCES_WORKSHEET).get_all_records()

    face_library = [clean_row(row) for row in face_library]
    occurrences = [clean_row(row) for row in occurrences]

    return face_library, occurrences


def get_confirmed_people(face_library_rows):
    """Return Face ID -> confirmed person info."""
    confirmed = {}

    for row in face_library_rows:
        face_id = str(row.get(FACE_ID_COL, "")).strip()
        name = str(row.get(CONFIRMED_NAME_COL, "")).strip()
        status = str(row.get(CONFIRM_STATUS_COL, "")).strip().lower()
        notes = str(row.get(NOTES_COL, "")).strip()

        if not face_id or not name or status not in APPROVED_VALUES:
            continue

        confirmed[face_id] = {
            "face_id": face_id,
            "name": name,
            "notes": notes,
        }

    return confirmed


face_library_rows, occurrence_rows = read_sheets(creds)
confirmed_people = get_confirmed_people(face_library_rows)

print(f"Face Library rows: {len(face_library_rows)}")
print(f"Face Occurrence rows: {len(occurrence_rows)}")
print(f"Confirmed Face IDs: {len(confirmed_people)}")
print(confirmed_people)

## 5. Build Planned File Updates

This joins confirmed Face IDs to occurrence rows, then groups all confirmed people by Drive file.

In [ ]:
def build_file_updates(face_library_rows, occurrence_rows):
    """Join confirmed Face IDs to occurrence rows and group by Drive file."""
    confirmed_people = get_confirmed_people(face_library_rows)
    file_updates = {}

    for row in occurrence_rows:
        face_id = str(row.get(FACE_ID_COL, "")).strip()
        if face_id not in confirmed_people:
            continue

        drive_link = row.get(DRIVE_LINK_COL, "")
        drive_id = parse_drive_file_id(drive_link)

        if not drive_id:
            print(f"Skipping row with missing/bad Drive link: {drive_link}")
            continue

        file_name = str(row.get(FILE_NAME_COL, "")).strip()
        media_type = str(row.get(MEDIA_TYPE_COL, "")).strip()

        if drive_id not in file_updates:
            file_updates[drive_id] = {
                "drive_id": drive_id,
                "file_name": file_name,
                "media_type": media_type,
                "people": {},
            }

        person = confirmed_people[face_id]

        if face_id not in file_updates[drive_id]["people"]:
            file_updates[drive_id]["people"][face_id] = {
                "face_id": face_id,
                "name": person["name"],
                "notes": person["notes"],
                "mentions": [],
            }

        file_updates[drive_id]["people"][face_id]["mentions"].append({
            "start_time": str(row.get(START_TIME_COL, "")).strip(),
            "end_time": str(row.get(END_TIME_COL, "")).strip(),
            "confidence": parse_float_or_none(row.get(CONFIDENCE_COL, "")),
        })

    return file_updates


file_updates = build_file_updates(face_library_rows, occurrence_rows)

print(f"Prepared updates for {len(file_updates)} Drive file(s).")
for drive_id, update in list(file_updates.items())[:10]:
    names = sorted([person["name"] for person in update["people"].values()])
    print(f"{update.get('file_name') or drive_id}: {', '.join(names)}")

## 6. Drive Description Writeback Functions

These functions preserve existing human-written descriptions and replace only the managed metadata block.

In [ ]:
def build_metadata(file_update):
    """Build metadata JSON to store in Drive file description."""
    people = list(file_update["people"].values())
    people = sorted(people, key=lambda p: (p["name"].lower(), p["face_id"]))

    return {
        "schema": "unlabeled-media-tagger",
        "version": 2,
        "source": "human_confirmed_spreadsheet",
        "updated_at": datetime.now(timezone.utc).isoformat(),
        "people": people,
    }


def get_current_description(drive_service, file_id):
    metadata = (
        drive_service.files()
        .get(fileId=file_id, fields="id,name,description")
        .execute()
    )
    return metadata.get("description", "")


def update_description(drive_service, file_id, description):
    return (
        drive_service.files()
        .update(
            fileId=file_id,
            body={"description": description},
            fields="id,name,description",
        )
        .execute()
    )


def write_updates(creds, file_updates):
    drive_service = build("drive", "v3", credentials=creds)

    print(f"Prepared updates for {len(file_updates)} Drive file(s).")

    for i, (drive_id, update) in enumerate(file_updates.items(), start=1):
        names = sorted([p["name"] for p in update["people"].values()])
        file_name = update.get("file_name") or drive_id

        print(f"[{i}/{len(file_updates)}] {file_name}: {', '.join(names)}")

        if DRY_RUN:
            continue

        existing_description = get_current_description(drive_service, drive_id)
        metadata = build_metadata(update)
        new_description = build_description(existing_description, metadata)
        update_description(drive_service, drive_id, new_description)

    if DRY_RUN:
        print("\nDRY_RUN=True. No Drive files were updated.")
        print("Set DRY_RUN=False after reviewing the planned updates.")

## 7. Run Dry Run / Writeback

Run this cell with `DRY_RUN = True` first. If the output looks correct, change `DRY_RUN = False` in the config cell and run this cell again.

In [ ]:
write_updates(creds, file_updates)

## Notes

- Only rows with a non-empty `Confirmed Name` and approved `Confirm Status` are written.
- Approved statuses are: `yes`, `y`, `true`, `approved`, `confirmed`.
- Existing human-written Drive descriptions are preserved.
- Previous `<unlabeled-media-tagger>...</unlabeled-media-tagger>` blocks are replaced, not duplicated.
- The notebook does not need face screenshots/contact sheets to write descriptions; those are for human review.